# SU(2) Hamiltonian-to-circuit experiments

Focused notebook for the current research loop: train the single-gate SU(2) diffusion sampler, synthesize Hamiltonian targets with two-CZ templates, and refine generated local-gate proposals into concrete circuit decompositions.

Current default path: uniform generated `SU(2)` search plus local `SU(2)` refinement. The learned slot-label prior cells are kept as diagnostics and are skipped unless explicitly enabled.

In [ ]:
# Run this in Colab or any fresh remote runtime before importing su2diffusion.
# Change BRANCH when testing a PR branch; use "main" after merge.
BRANCH = "codex/hamiltonian-workflow-cleanup"

!pip install --no-cache-dir --force-reinstall --no-deps git+https://github.com/joe-singh/su2diffusion.git@{BRANCH}

In [ ]:
from dataclasses import replace

import torch

from su2diffusion import (
    HamiltonianPriorTrainConfig,
    HamiltonianSupervisedTrainConfig,
    center_names_for_config,
    centers_for_config,
    get_experiment_config,
    make_hamiltonian_target,
    make_random_pauli_hamiltonian_targets,
    plot_experiment_report,
    plot_hamiltonian_budget_refinement,
    plot_hamiltonian_mixture_refinement,
    plot_hamiltonian_solution_dataset,
    plot_hamiltonian_prior_mixture,
    plot_hamiltonian_prior_search,
    plot_hamiltonian_seed_ablation,
    plot_hamiltonian_suite,
    plot_hamiltonian_supervised_split_result,
    plot_hamiltonian_two_entangler_benchmark,
    print_center_mass_table,
    print_conditional_label_table,
    print_diagnostics_table,
    print_hamiltonian_budget_refinement_summary,
    print_hamiltonian_mixture_refinement,
    print_hamiltonian_mixture_refinement_summary,
    print_hamiltonian_solution_dataset,
    print_hamiltonian_solution_dataset_summary,
    print_hamiltonian_prior_mixture_summary,
    print_hamiltonian_prior_search,
    print_hamiltonian_prior_search_summary,
    print_hamiltonian_seed_ablation,
    print_hamiltonian_seed_ablation_summary,
    print_hamiltonian_suite,
    print_hamiltonian_suite_summary,
    print_hamiltonian_supervised_split_summary,
    print_hamiltonian_target,
    print_hamiltonian_two_entangler_benchmark,
    print_hamiltonian_two_entangler_summary,
    refine_hamiltonian_prior_mixture,
    refine_hamiltonian_prior_mixture_budget_sweep,
    generate_hamiltonian_solution_dataset,
    run_experiment,
    run_hamiltonian_prior_mixture_sweep,
    run_hamiltonian_prior_search_benchmark,
    run_hamiltonian_seed_ablation,
    run_hamiltonian_suite_benchmark,
    run_hamiltonian_supervised_split_baseline,
    run_hamiltonian_two_entangler_benchmark,
    train_hamiltonian_slot_prior,
)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## Train single-qubit Clifford diffusion

This is the only default diffusion training cell. It produces local SU(2) gates used by the Hamiltonian synthesis/search cells below.

In [ ]:
CONFIG_NAME = "baseline-clifford-cond"
EVAL_COUNT = 1000

config = get_experiment_config(CONFIG_NAME)
config = replace(config, sample_count=EVAL_COUNT, reference_count=EVAL_COUNT)
config

In [ ]:
result = run_experiment(config, device=device)

center_names = center_names_for_config(result.config.data)
print_diagnostics_table(result.diagnostics)
print()
print_center_mass_table(result.diagnostics, center_names=center_names)
if result.conditional_diagnostics is not None:
    print()
    print_conditional_label_table(result.conditional_diagnostics, center_names=center_names)
plot_experiment_report(result)

In [ ]:
# Local gates used by Hamiltonian synthesis baselines.
exact_gates = centers_for_config(result.config.data, device=device)
local_gates = result.generated_stochastic
local_labels = [center_names[int(i)] for i in result.stochastic_labels]

print(f"exact Clifford gates: {exact_gates.shape[0]}")
print(f"generated local gates: {local_gates.shape[0]}")

# Optional diagnostics from prior branches. Leave False for the default synthesis workflow.
RUN_LABEL_PRIOR_DIAGNOSTICS = False
print("label-prior diagnostics:", RUN_LABEL_PRIOR_DIAGNOSTICS)

## Single Hamiltonian Target

Build a two-qubit Hamiltonian from Pauli-string terms, exponentiate it to `U(t)`, then synthesize against that unitary with the two-CZ template.

In [ ]:
hamiltonian_target = make_hamiltonian_target(
    [
        ("XI", 0.35),   # X0 = X tensor I
        ("IZ", -0.22),  # Z1 = I tensor Z
        ("XX", 0.18),   # X0 X1 = X tensor X
        ("ZZ", 0.12),   # Z0 Z1 = Z tensor Z
    ],
    time=0.8,
    name="pauli-demo",
    device=device,
)

hamiltonian_benchmark = run_hamiltonian_two_entangler_benchmark(
    hamiltonian_target,
    clifford_gates=exact_gates,
    clifford_labels=center_names,
    generated_gates=local_gates,
    generated_labels=local_labels,
    perturb_scale=0.12,
    n_random_candidates=200_000,
    n_analytic_gates=1024,
    n_haar_gates=1024,
    top_k=5,
    seed=404,
    keep_fidelities=False,
)

print_hamiltonian_target(hamiltonian_target)
print()
print_hamiltonian_two_entangler_benchmark(hamiltonian_benchmark)
print()
print_hamiltonian_two_entangler_summary(hamiltonian_benchmark)
plot_hamiltonian_two_entangler_benchmark(hamiltonian_benchmark)

## Hamiltonian Suite

Run the same two-CZ search over a small family of random Pauli Hamiltonians. This is the main benchmark for proposal quality.

In [ ]:
hamiltonian_targets = make_random_pauli_hamiltonian_targets(
    n_targets=12,
    terms=("XI", "IZ", "XX", "ZZ"),
    coefficient_scale=0.35,
    time=0.8,
    seed=505,
    device=device,
)

hamiltonian_suite = run_hamiltonian_suite_benchmark(
    hamiltonian_targets,
    clifford_gates=exact_gates,
    clifford_labels=center_names,
    generated_gates=local_gates,
    generated_labels=local_labels,
    perturb_scale=0.12,
    n_random_candidates=100_000,
    n_analytic_gates=1024,
    n_haar_gates=1024,
    top_k=5,
    seed=606,
    keep_fidelities=False,
)

print_hamiltonian_suite(hamiltonian_suite)
print()
print_hamiltonian_suite_summary(hamiltonian_suite)
plot_hamiltonian_suite(hamiltonian_suite)

## Default Hamiltonian Synthesis Baseline

Use uniform generated `SU(2)` local-gate search, then refine the best two-CZ candidate directly on the `SU(2)^6` product manifold. This is the current main Hamiltonian-to-circuit baseline.

In [ ]:
hamiltonian_solution_dataset = generate_hamiltonian_solution_dataset(
    hamiltonian_targets,
    clifford_gates=exact_gates,
    clifford_labels=center_names,
    generated_gates=local_gates,
    generated_labels=local_labels,
    perturb_scale=0.12,
    n_random_candidates=100_000,
    n_analytic_gates=1024,
    n_haar_gates=1024,
    top_k=5,
    seed=707,
    refinement_steps=200,
    refinement_lr=0.05,
    fidelity_threshold=0.0,
)

print_hamiltonian_solution_dataset(hamiltonian_solution_dataset)
print()
print_hamiltonian_solution_dataset_summary(hamiltonian_solution_dataset)
plot_hamiltonian_solution_dataset(hamiltonian_solution_dataset)

## Supervised Hamiltonian Baseline

Train a simple deterministic MLP baseline on the generated solution stacks, then evaluate on both the training Hamiltonians and a held-out Hamiltonian set. This is not diffusion yet; it checks whether `(H, t) -> six local SU(2) gates` generalizes at all.

In [ ]:
if RUN_LABEL_PRIOR_DIAGNOSTICS:
    supervised_config = HamiltonianSupervisedTrainConfig(
        hidden=256,
        num_steps=1000,
        lr=1e-3,
        weight_decay=1e-4,
        seed=0,
    )

    heldout_hamiltonian_targets = make_random_pauli_hamiltonian_targets(
        n_targets=12,
        terms=("XI", "IZ", "XX", "ZZ"),
        coefficient_scale=0.35,
        time=0.8,
        seed=808,
        device=device,
    )

    hamiltonian_supervised_result = run_hamiltonian_supervised_split_baseline(
        hamiltonian_solution_dataset,
        heldout_hamiltonian_targets,
        config=supervised_config,
        device=device,
        refine=True,
        refinement_steps=100,
        refinement_lr=0.05,
    )

    print(f"final supervised loss: {hamiltonian_supervised_result.train.losses[-1]:.6f}")
    print_hamiltonian_supervised_split_summary(hamiltonian_supervised_result)
    plot_hamiltonian_supervised_split_result(hamiltonian_supervised_result)
else:
    print("Skipping supervised/prior diagnostics. Set RUN_LABEL_PRIOR_DIAGNOSTICS = True to run this cell.")

## Hamiltonian Seed Ablation

Compare the supervised MLP, generated-search, Clifford-search, and Haar seeds under the same local refinement budget. The key question is not only final fidelity, but which seed reaches high fidelity fastest.

In [ ]:
if RUN_LABEL_PRIOR_DIAGNOSTICS:
    heldout_hamiltonian_suite = run_hamiltonian_suite_benchmark(
        heldout_hamiltonian_targets,
        clifford_gates=exact_gates,
        clifford_labels=center_names,
        generated_gates=local_gates,
        generated_labels=local_labels,
        perturb_scale=0.12,
        n_random_candidates=100_000,
        n_analytic_gates=1024,
        n_haar_gates=1024,
        top_k=5,
        seed=909,
        keep_fidelities=False,
    )

    hamiltonian_seed_ablation = run_hamiltonian_seed_ablation(
        heldout_hamiltonian_targets,
        hamiltonian_supervised_result.heldout.predicted_stacks,
        heldout_hamiltonian_suite,
        clifford_gates=exact_gates,
        generated_gates=local_gates,
        threshold=0.99,
        refinement_steps=100,
        refinement_lr=0.05,
        seed=1001,
    )

    print_hamiltonian_seed_ablation(hamiltonian_seed_ablation)
    print()
    print_hamiltonian_seed_ablation_summary(hamiltonian_seed_ablation)
    plot_hamiltonian_seed_ablation(hamiltonian_seed_ablation)
else:
    print("Skipping seed ablation diagnostics.")

## Hamiltonian Learned-Prior Search

Experimental diagnostic: train a slot-label prior from Hamiltonian features to labels found in the refined solution dataset, then use that prior to bias generated-gate search on held-out Hamiltonians. This is skipped by default because the hard benchmark showed it is not robust enough to be the main synthesis path.

In [ ]:
if RUN_LABEL_PRIOR_DIAGNOSTICS:
    prior_config = HamiltonianPriorTrainConfig(
        hidden=128,
        num_steps=500,
        lr=1e-3,
        weight_decay=1e-4,
        seed=0,
    )

    hamiltonian_prior = train_hamiltonian_slot_prior(
        hamiltonian_solution_dataset,
        center_names,
        config=prior_config,
        device=device,
    )

    hamiltonian_prior_search = run_hamiltonian_prior_search_benchmark(
        hamiltonian_prior,
        heldout_hamiltonian_targets,
        local_gates=local_gates,
        local_labels=local_labels,
        n_candidates=100_000,
        top_k=5,
        seed=1101,
        keep_fidelities=False,
    )

    print(f"final prior loss: {hamiltonian_prior.losses[-1]:.6f}")
    print(f"train slot-label accuracy: {hamiltonian_prior.train_accuracy:.1%}")
    print_hamiltonian_prior_search(hamiltonian_prior_search)
    print()
    print_hamiltonian_prior_search_summary(hamiltonian_prior_search)
    plot_hamiltonian_prior_search(hamiltonian_prior_search)
else:
    print("Skipping learned-prior diagnostics.")

## Hamiltonian Prior-Mixture Sweep

Blend the learned slot-label prior with uniform exploration. `alpha=0` is pure uniform label sampling; `alpha=1` is the pure learned prior.

In [ ]:
if RUN_LABEL_PRIOR_DIAGNOSTICS:
    hamiltonian_prior_mixture = run_hamiltonian_prior_mixture_sweep(
        hamiltonian_prior,
        heldout_hamiltonian_targets,
        local_gates=local_gates,
        local_labels=local_labels,
        alphas=(0.0, 0.25, 0.5, 0.75, 1.0),
        n_candidates=100_000,
        top_k=5,
        seed=1201,
        keep_fidelities=False,
    )

    print_hamiltonian_prior_mixture_summary(hamiltonian_prior_mixture)
    plot_hamiltonian_prior_mixture(hamiltonian_prior_mixture)
else:
    print("Skipping prior-mixture diagnostics.")

## Hamiltonian Prior-Mixture Refinement

Refine the best candidate from each alpha and measure both final fidelity and the number of optimizer steps needed to reach the target threshold.

In [ ]:
if RUN_LABEL_PRIOR_DIAGNOSTICS:
    hamiltonian_mixture_refinement = refine_hamiltonian_prior_mixture(
        hamiltonian_prior_mixture,
        local_gates=local_gates,
        threshold=0.99,
        refinement_steps=100,
        refinement_lr=0.05,
    )

    print_hamiltonian_mixture_refinement(hamiltonian_mixture_refinement)
    print()
    print_hamiltonian_mixture_refinement_summary(hamiltonian_mixture_refinement)
    plot_hamiltonian_mixture_refinement(hamiltonian_mixture_refinement)
else:
    print("Skipping prior-mixture refinement diagnostics.")

## Harder Hamiltonian Benchmark

Optional diagnostic: use more Pauli terms, more held-out targets, and constrained refinement budgets to stress-test whether the Hamiltonian-conditioned label prior transfers out of distribution. The current result is that uniform generated search is the safer default.

In [ ]:
if RUN_LABEL_PRIOR_DIAGNOSTICS:
    HARD_TERMS = (
        "XI", "YI", "ZI",
        "IX", "IY", "IZ",
        "XX", "XY", "XZ",
        "YX", "YY", "YZ",
        "ZX", "ZY", "ZZ",
    )

    hard_hamiltonian_targets = make_random_pauli_hamiltonian_targets(
        n_targets=24,
        terms=HARD_TERMS,
        coefficient_scale=0.45,
        time=1.1,
        seed=1301,
        device=device,
    )

    hard_hamiltonian_prior_mixture = run_hamiltonian_prior_mixture_sweep(
        hamiltonian_prior,
        hard_hamiltonian_targets,
        local_gates=local_gates,
        local_labels=local_labels,
        alphas=(0.0, 0.25, 0.5, 0.75, 1.0),
        n_candidates=50_000,
        top_k=5,
        seed=1401,
        keep_fidelities=False,
    )

    print_hamiltonian_prior_mixture_summary(hard_hamiltonian_prior_mixture)
    plot_hamiltonian_prior_mixture(hard_hamiltonian_prior_mixture)
else:
    print("Skipping hard prior-transfer benchmark.")

## Harder Benchmark With Refinement Budgets

Refine the hard benchmark under small optimizer budgets. If the learned prior matters for synthesis efficiency, it should show up here as higher fidelity or higher success at low budgets.

In [ ]:
if RUN_LABEL_PRIOR_DIAGNOSTICS:
    hard_hamiltonian_budget_refinement = refine_hamiltonian_prior_mixture_budget_sweep(
        hard_hamiltonian_prior_mixture,
        local_gates=local_gates,
        budgets=(5, 10, 20, 50),
        refinement_lr=0.05,
        threshold=0.99,
    )

    print_hamiltonian_budget_refinement_summary(hard_hamiltonian_budget_refinement)
    plot_hamiltonian_budget_refinement(hard_hamiltonian_budget_refinement)
else:
    print("Skipping hard budgeted-refinement diagnostic.")